In [30]:
import pandas as pd
import os
import numpy as np
from sqlalchemy import text , create_engine
from warnings import filterwarnings
filterwarnings('ignore')

In [40]:
engine = create_engine(
    f"mysql+mysqlconnector://{os.environ['DB_USER']}:{os.environ['DB_PASSWORD']}@{os.environ['DB_HOST']}/{os.environ['DB_NAME']}"
)

with engine.connect() as conn:
    result = conn.execute(text("SHOW TABLES"))
    tables = [row[0] for row in result]

print("Total Tables:", len(tables))
print("Table Names:")
for table in tables:
    print("-", table)


Total Tables: 1
Table Names:
- rapido_july2025_data


In [41]:
for table in tables:
    #    print(f"\n Table: {table}")
       query = text(f"SELECT COUNT(*) FROM {table}")
       df = pd.read_sql_query(query, engine)
       print(f"{table}", df.iloc[0,0])
       display(pd.read_sql(f"SELECT * FROM {table} LIMIT 5", engine))

rapido_july2025_data 30000


,Booking_ID,Booking_Status,Booking_Value,Customer_ID,Driver_ID,Pickup_Location,Drop_Location,Ride_Distance(km),Ride_Time(min),Date,...,Driver_Rating,Canceled_Rides_by_Customer,Canceled_Rides_by_Driver,Incomplete_Rides,Incomplete_Rides_Reason,Total_Bookings,Canceled_Bookings,Canceled_Percentage,V_TAT,C_TAT
0,RAP20250700001,Completed,170.39,CUST_2824,DR_902,Pune,Delhi,2.74,42,2025-07-02,...,4.3,0,0,0,,998,122,12.22,19,27
1,RAP20250700002,Incomplete,131.04,CUST_1409,DR_915,Chennai,Hyderabad,22.15,20,2025-07-09,...,4.9,0,0,1,Driver delayed,986,119,12.07,25,12
2,RAP20250700003,Completed,242.19,CUST_5506,DR_938,Delhi,Bengaluru,11.95,46,2025-07-13,...,4.7,0,0,0,,972,114,11.73,18,25
3,RAP20250700004,Completed,78.18,CUST_5012,DR_213,Pune,Hyderabad,9.08,24,2025-07-27,...,4.0,0,0,0,,961,112,11.65,11,6
4,RAP20250700005,Completed,159.33,CUST_4657,DR_783,Chennai,Bengaluru,22.97,40,2025-07-22,...,3.0,0,0,0,,963,115,11.94,22,25


In [42]:
df = pd.read_sql_query(text("select * from rapido_july2025_data") , engine)

In [43]:
print(f"{df.info()}")
print("-"* 40)
print(f"null check : \n{df.isnull().sum()}")
print("-"* 40)
print(f"duplicate data : {df.duplicated().sum()}")
print("-"* 40)
print(f"data shape : \n{df.shape}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Booking_ID                  30000 non-null  object 
 1   Booking_Status              30000 non-null  object 
 2   Booking_Value               30000 non-null  float64
 3   Customer_ID                 30000 non-null  object 
 4   Driver_ID                   30000 non-null  object 
 5   Pickup_Location             30000 non-null  object 
 6   Drop_Location               30000 non-null  object 
 7   Ride_Distance(km)           30000 non-null  float64
 8   Ride_Time(min)              30000 non-null  int64  
 9   Date                        30000 non-null  object 
 10  Time                        30000 non-null  object 
 11  Vehicle_Type                30000 non-null  object 
 12  Vehicle_Image               30000 non-null  object 
 13  Payment_Method              300

In [44]:
df = df.drop(columns=["Vehicle_Image" , "Booking_ID" , "Customer_ID" , "Driver_ID"])

In [45]:
# check unique values
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    vals = df[col].unique()
    if(len(vals) <=30):
        print(f"\n{col} ({len(vals)} unique): {sorted(vals)}")


Booking_Status (3 unique): ['Cancelled', 'Completed', 'Incomplete']

Pickup_Location (5 unique): ['Bengaluru', 'Chennai', 'Delhi', 'Hyderabad', 'Pune']

Drop_Location (5 unique): ['Bengaluru', 'Chennai', 'Delhi', 'Hyderabad', 'Pune']

Vehicle_Type (2 unique): ['Auto', 'Bike']

Payment_Method (4 unique): ['Card', 'Cash', 'UPI', 'Wallet']

Incomplete_Rides_Reason (6 unique): ['', 'Customer cancelled early', 'Customer not found', 'Driver delayed', 'Network issue', 'Weather issue']


In [46]:
df['Incomplete_Rides_Reason'].value_counts()

Incomplete_Rides_Reason
                            28200
Network issue                 380
Customer cancelled early      367
Driver delayed                362
Customer not found            356
Weather issue                 335
Name: count, dtype: int64

* convert Incomplete_Rides_Reason "" to "Ride Complate" as 80 % of data are empty string here(not null) and we can't drop it 

In [47]:
df['Incomplete_Rides_Reason'] = (
    df['Incomplete_Rides_Reason']
    .replace('', 'Ride Complate')
)

In [48]:
df['Incomplete_Rides_Reason'].value_counts()

Incomplete_Rides_Reason
Ride Complate               28200
Network issue                 380
Customer cancelled early      367
Driver delayed                362
Customer not found            356
Weather issue                 335
Name: count, dtype: int64

In [60]:
df['Date'] = pd.to_datetime(df['Date'] , errors='coerce')
df['current_date_'] = pd.to_datetime(df['Date']).dt.day
df['hour'] = pd.to_datetime(df['Time']).dt.hour
df['day'] = pd.to_datetime(df['Date']).dt.day_of_week
df['day_name'] = pd.to_datetime(df['Date']).dt.day_name()

In [62]:
# remove whitespace from object columns
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].str.strip()

In [63]:
df.columns

Index(['Booking_Status', 'Booking_Value', 'Pickup_Location', 'Drop_Location',
       'Ride_Distance(km)', 'Ride_Time(min)', 'Date', 'Time', 'Vehicle_Type',
       'Payment_Method', 'Customer_Rating', 'Driver_Rating',
       'Canceled_Rides_by_Customer', 'Canceled_Rides_by_Driver',
       'Incomplete_Rides', 'Incomplete_Rides_Reason', 'Total_Bookings',
       'Canceled_Bookings', 'Canceled_Percentage', 'V_TAT', 'C_TAT',
       'current_date_', 'hour', 'day', 'day_name'],
      dtype='object')

In [64]:
df.to_sql(name="clean_dataset",
          con=engine , 
          if_exists="replace" , 
          index=False,
          chunksize=500)
print("clean dataset save in mysql")

clean dataset save in mysql


In [65]:
df_verify = pd.read_sql("SELECT * FROM clean_dataset LIMIT 5", engine)
display(df_verify)

,Booking_Status,Booking_Value,Pickup_Location,Drop_Location,Ride_Distance(km),Ride_Time(min),Date,Time,Vehicle_Type,Payment_Method,...,Incomplete_Rides_Reason,Total_Bookings,Canceled_Bookings,Canceled_Percentage,V_TAT,C_TAT,current_date_,hour,day,day_name
0,Completed,170.39,Pune,Delhi,2.74,42,2025-07-02,09:43,Bike,UPI,...,Ride Complate,998,122,12.22,19,27,2,9,2,Wednesday
1,Incomplete,131.04,Chennai,Hyderabad,22.15,20,2025-07-09,17:34,Auto,Cash,...,Driver delayed,986,119,12.07,25,12,9,17,2,Wednesday
2,Completed,242.19,Delhi,Bengaluru,11.95,46,2025-07-13,23:54,Bike,Card,...,Ride Complate,972,114,11.73,18,25,13,23,6,Sunday
3,Completed,78.18,Pune,Hyderabad,9.08,24,2025-07-27,17:41,Bike,Wallet,...,Ride Complate,961,112,11.65,11,6,27,17,6,Sunday
4,Completed,159.33,Chennai,Bengaluru,22.97,40,2025-07-22,22:52,Bike,Cash,...,Ride Complate,963,115,11.94,22,25,22,22,1,Tuesday


In [66]:
original_count = pd.read_sql("SELECT COUNT(*) AS cnt FROM clean_dataset", engine)
print(f"✓ Rows in MySQL  : {original_count['cnt'][0]}")
print(f"✓ Rows in df     : {len(df)}")
print(f"✓ Match          : {original_count['cnt'][0] == len(df)}")

✓ Rows in MySQL  : 30000
✓ Rows in df     : 30000
✓ Match          : True
